In [ ]:

import speech_recognition as sr
import threading
import queue

from datetime import datetime



LANGUAGE = "ur-PK"          
ENERGY_THRESHOLD = 300      
PAUSE_THRESHOLD = 0.8       
PHRASE_LIMIT = None         


def listen_worker(recognizer: sr.Recognizer, source: sr.Microphone,
                  result_queue: queue.Queue, stop_event: threading.Event):
    
    print("  [Mic open — start speaking]\n")
    while not stop_event.is_set():  
        try:
            audio = recognizer.listen(source,timeout=1, phrase_time_limit=PHRASE_LIMIT)
            result_queue.put(audio)
            
        except sr.WaitTimeoutError:
            pass  # No speech detected in this window; keep looping


def recognise_worker(recognizer: sr.Recognizer, result_queue: queue.Queue,
                     stop_event: threading.Event):
    """Pulls audio chunks from the queue and converts them to text."""
    while not stop_event.is_set() or not result_queue.empty():
        try:
            audio = result_queue.get(timeout=1)
        except queue.Empty:
            continue

        try:
            text = recognizer.recognize_google(audio, language=LANGUAGE)
            ts = datetime.now().strftime("%H:%M:%S")
            print(f"[{ts}] {text}")
        except sr.UnknownValueError:
            print("         ... (unclear audio)")
        except sr.RequestError as e:
            print(f"  [API error: {e}]")
        finally:
            result_queue.task_done()


def main():
    recognizer = sr.Recognizer()
    recognizer.energy_threshold = ENERGY_THRESHOLD
    recognizer.pause_threshold = PAUSE_THRESHOLD
    recognizer.dynamic_energy_threshold = True   # Auto-adjusts to ambient noise

    print("━" * 50)
    print("  🎙  Live Speech Recognition")
    print(f"      Language : {LANGUAGE}")
    print("      Press Ctrl+C to stop")
    print("━" * 50)

    result_queue: queue.Queue = queue.Queue()
    stop_event = threading.Event()

    with sr.Microphone() as source:
        print("  Adjusting for ambient noise… ", end="", flush=True)
        recognizer.adjust_for_ambient_noise(source, duration=1)
        print("done.")

        # Thread 1
        listener = threading.Thread(
            target=listen_worker,
            args=(recognizer, source, result_queue, stop_event),
            daemon=True,
        )
        # Thread 2
        recogniser = threading.Thread(
            target=recognise_worker,
            args=(recognizer, result_queue, stop_event),
            daemon=True,
        )

        listener.start()
        recogniser.start()

        try:
            listener.join()
        except KeyboardInterrupt:
            print("\n\n  Stopping…")
            stop_event.set()

    result_queue.join()          
    print("  Done. Goodbye!")


if __name__ == "__main__":
    main()

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🎙  Live Speech Recognition
      Language : ur-PK
      Press Ctrl+C to stop
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Adjusting for ambient noise… done.
  [Mic open — start speaking]

         ... (unclear audio)
         ... (unclear audio)
[17:23:46] میرا نام صہیب ہے
         ... (unclear audio)
         ... (unclear audio)
         ... (unclear audio)
         ... (unclear audio)
         ... (unclear audio)
[17:24:04] نہیں یار


In [1]:
from evaluate import load
